# 1. Importing Necessary Libraries and datasets and EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter('ignore')

import seaborn as sns
import sys
import itertools
import gc

from sklearn.model_selection import train_test_split

import csv
from collections import OrderedDict
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

import datetime

In [ ]:
train = pd.read_csv("../input/novozymes-enzyme-stability-prediction/train.csv")
test = pd.read_csv("../input/novozymes-enzyme-stability-prediction/test.csv")

In [ ]:
base = 'VPVNPEPDATSVENVALKTGSGDSQSDPIKADLEVKGQSALPFDVDCWAILCKGAPNVLQRVNEKTKNSNRDRSGANKGPFKDPQKWGIKALPPKNPSWSAQDFKSPEEYAFASSLQGGTNAILAPVNLASQNSQGGVLNGFYSANKVAQFDPSKPQQTKGTWFQITKFTGAAGPYCKALGSNDKSVCDKNKNIAGDWGFDPAKWAYQYDEKNNKFNYVGK'

def compare_columns(df1, df2):
    df1_columns_set = set(df1.columns)
    df2_columns_set = set(df2.columns)
    print('df1_columns_set - df2_columns_set :', df1_columns_set - df2_columns_set)
    print('df2_columns_set - df1_columns_set :', df2_columns_set - df1_columns_set)
    
def get_test_mutation(row):
    for i,(a,b) in enumerate(zip(row.protein_sequence,base)):
        if a!=b: break
    row['wildtype'] = base[i]
    row['mutation'] = row.protein_sequence[i]
    row['position'] = i+1
    return row

In [ ]:
test = test.apply(get_test_mutation,axis=1)
train = train.apply(get_test_mutation,axis=1)

In [ ]:
compare_columns(train, test)

In [ ]:
train.describe()

In [ ]:
def plot_two(df, feat1_name, feat2_name, shape):
    plt.figure(figsize=shape)
    sns.regplot(data=df, x=feat1_name, y=feat2_name, scatter_kws={'alpha':0.2})
    plt.title(feat1_name+' vs '+feat2_name, fontsize=14)
    plt.show()

In [ ]:
feat1_name, feat2_name = 'pH', 'tm'
plot_two(train, feat1_name, feat2_name, (12,6))

In [ ]:
train.info()

In [ ]:
test.info()

In [ ]:
train.head(5)

# 2. Data Preprocessing

In [ ]:
def outlier_replace(data, col_name, q1=0.001, q3=0.98):
    quartile1 = data[col_name].quantile(q1)
    quartile3 = data[col_name].quantile(q3)
    interquantile_range = quartile3 - quartile1
    up_limit = quartile3 + 1.5 * interquantile_range
    low_limit = quartile1 - 1.5 * interquantile_range
    data.loc[(data[col_name] < low_limit), col_name] = low_limit
    data.loc[(data[col_name] > up_limit), col_name] = up_limit

In [ ]:
outlier_replace(train,'pH')
outlier_replace(test, 'pH')

In [ ]:
feat1_name, feat2_name = 'pH', 'tm'
plot_two(train, feat1_name, feat2_name, (12,6))

In [ ]:
for col in train:
    train[col] = train[col].fillna(train[col].mode()[0])

In [ ]:
data = train.drop(columns=['tm'])
total_data = pd.concat([data,test])

In [ ]:
total_data.info()

In [ ]:
total_data.head(5)

In [ ]:
total_data['split'] = total_data['data_source'].str.split('/')

In [ ]:
total_data['org'] = total_data['split'].map(lambda x: x[0].strip())
total_data['org_code'] = LabelEncoder().fit_transform(total_data['org'])

In [ ]:
total_data['suborg'] = total_data['split'].map(lambda x: x[1].strip() if len(x) > 1 else x[0].strip())
total_data['suborg_code'] = LabelEncoder().fit_transform(total_data['suborg'])

In [ ]:
total_data.head(5)

In [ ]:
total_data=total_data.sort_values(by='protein_sequence')

In [ ]:
total_data['protein_sequence_no'] = LabelEncoder().fit_transform(total_data['protein_sequence'])

In [ ]:
total_data=total_data.sort_values(by='seq_id')

In [ ]:
total_data["protein_sequence_len"] = total_data["protein_sequence"].apply(lambda x: len(x))

In [ ]:
base = 'VPVNPEPDATSVENVALKTGSGDSQSDPIKADLEVKGQSALPFDVDCWAILCKGAPNVLQRVNEKTKNSNRDRSGANKGPFKDPQKWGIKALPPKNPSWSAQDFKSPEEYAFASSLQGGTNAILAPVNLASQNSQGGVLNGFYSANKVAQFDPSKPQQTKGTWFQITKFTGAAGPYCKALGSNDKSVCDKNKNIAGDWGFDPAKWAYQYDEKNNKFNYVGK'

In [ ]:
# source: https://www.kaggle.com/competitions/novozymes-enzyme-stability-prediction/discussion/354783
import Levenshtein

def get_mutation_info(_row, _wildtype=base):
    terminology_map = {"replace":"substitution", "insert":"insertion", "delete":"deletion"}
    req_edits = Levenshtein.editops(_wildtype, _row["protein_sequence"])
    _row["n_edits"] = len(req_edits)

    if _row["n_edits"]==0:
        _row["edit_type"] = _row["edit_idx"] = _row["old_aa"] = _row["new_aa"] = pd.NA
    else:
        _row["edit_type"] = terminology_map[req_edits[0][0]]
        _row["edit_idx"] = req_edits[0][1]
        _row["old_aa"] = _wildtype[_row["edit_idx"]]
        _row["new_aa"] = _row["protein_sequence"][req_edits[0][2]] if _row["edit_type"]!="deletion" else pd.NA
    return _row

def revert_to_wildtype(protein_sequence, edit_type, edit_idx, old_aa, new_aa):
    if pd.isna(edit_type):
        return protein_sequence
    elif edit_type!="insertion":
        new_wildtype_base = protein_sequence[:edit_idx]
        if edit_type=="deletion":
            new_wildtype=new_wildtype_base+old_aa+protein_sequence[edit_idx:]
        else:
            new_wildtype=new_wildtype_base+old_aa+protein_sequence[edit_idx+1:]
    else:
        new_wildtype=protein_sequence[:edit_idx]+old_aa+protein_sequence[edit_idx:]
    return new_wildtype




#helper function
def read_list_from_file(list_file):
    with open(list_file) as f:
        lines  = f.readlines()
    return lines

In [ ]:
total_data = total_data.apply(get_mutation_info, axis=1)
total_data.loc[total_data.edit_type.isna(), 'edit_type'] = 'nothing'

In [ ]:
from Bio.SubsMat import MatrixInfo

sub_scores = []
sub_mat = MatrixInfo.blosum100
for i in range(len(total_data)):
    mut_type = total_data.edit_type.values[i]
    if mut_type == 'substitution':
        try:
            sub_score = sub_mat[(total_data.old_aa.values[i], total_data.new_aa.values[i])]
        except KeyError:
            sub_score = sub_mat[(total_data.new_aa.values[i], total_data.old_aa.values[i])]
    elif mut_type == 'nothing':
        sub_score = 0
    else:
        sub_score = -10
    sub_scores.append(sub_score)

In [ ]:
cap_sub_score_zero = False

total_data['sub_score'] = sub_scores
if cap_sub_score_zero:
    total_data.loc[total_data['sub_score'] > 0, 'sub_score'] = 0
total_data['score_adj'] = [ 1 - (1 / (1+np.exp(-x/3))) for x in sub_scores]
# total_data['b_factor_adj'] = total_data['b_factor'] * total_data['score_adj'] 

In [ ]:
total_data.head(5)

In [ ]:
total_data.info()

In [ ]:
from scipy.sparse import csr_matrix

# total_data = total_data[total_data["protein_sequence_len"]<=221]
# total_data.reset_index(inplace=True)
# data["protein_sequence_len"] = data["protein_sequence"].apply(lambda x: len(x))
# data = data[data["protein_sequence_len"]<=221]
# data.reset_index(inplace=True)
sequences = [list(string) for string in total_data["protein_sequence"].values.tolist()]
sequences_train = pd.DataFrame(sequences)
sequences_train.head()

In [ ]:
sequences_train = sequences_train.apply(LabelEncoder().fit_transform)
for i in range(221):
    total_data[i+18] = sequences_train[i]
total_data.head()

In [ ]:
total_data = total_data.drop(columns=['protein_sequence', 'data_source','split'])

In [ ]:
total_data = pd.get_dummies(total_data)

In [ ]:
total_data.info()

In [ ]:
import re
total_data = total_data.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', str(x)))

In [ ]:
total_data.head(5)

In [ ]:
data, df_test = total_data[:len(data)], total_data[len(data):]

In [ ]:
data.info()

In [ ]:
df_test.info()

In [ ]:
target = train['tm']
test_id = df_test['seq_id']

In [ ]:
target.head(5)

# 3.Modeling

In [ ]:
from sklearn.model_selection import RepeatedKFold, KFold, cross_val_score, train_test_split, GridSearchCV, RandomizedSearchCV

# 5 Fold Cross validation
kf = KFold(n_splits=5, shuffle=True)
cv_scores, cv_std = [], []

In [ ]:
# Creation of the RMSE metric:    
def rmse(model):
    return np.sqrt(-cross_val_score(model, data, target, scoring="neg_mean_squared_error", cv=kf))

In [ ]:
def apply_learning_algorithm(model):
    score = rmse(model)
    cv_scores.append(score.mean())
    cv_std.append(score.std())

In [ ]:
from lightgbm                import LGBMRegressor
from xgboost                 import XGBRegressor
from sklearn.svm             import SVR
from sklearn.metrics         import mean_squared_error, mean_absolute_error, mean_squared_log_error
# import warnings
# warnings.simplefilter('ignore')

# models = [LGBMRegressor(objective='regression',
#                         num_leaves=966,
#                         learning_rate=0.01, 
#                         n_estimators=720,
#                         max_bin = 55, 
#                         bagging_fraction = 0.8,
#                         bagging_freq = 5, 
#                         feature_fraction = 0.2319,
#                         feature_fraction_seed=9, 
#                         bagging_seed=9,
#                         min_data_in_leaf =6, 
#                         min_sum_hessian_in_leaf = 11),
#           SVR(kernel='rbf', C=100000, epsilon=0.01),
#           XGBRegressor(max_depth=7,learning_rate=0.01,
#                         n_estimators=2700,
#                         min_child_weight=0.5, 
#                         colsample_bytree=0.8, 
#                         subsample=0.8, 
#                         eta=0.5,
#                         seed=42)]

models = [LGBMRegressor(objective='regression',
                        num_leaves=166,
                        learning_rate=0.05, 
                        n_estimators=120,
                        max_bin = 55, 
                        bagging_fraction = 0.8,
                        bagging_freq = 5, 
                        feature_fraction = 0.2319,
                        feature_fraction_seed=9, 
                        bagging_seed=9,
                        min_data_in_leaf =6, 
                        min_sum_hessian_in_leaf = 11),
          SVR(kernel='rbf', C=10000, epsilon=0.05),
          XGBRegressor(max_depth=7,learning_rate=0.05,
                        n_estimators=700,
                        min_child_weight=0.5, 
                        colsample_bytree=0.8, 
                        subsample=0.8, 
                        eta=0.5,
                        seed=42)]

In [ ]:
model_names = ['LGBMRegressor','SupportVectorRegressor','XGBRegressor']

In [ ]:
for model in models:
    apply_learning_algorithm(model)

In [ ]:
cv_scores

In [ ]:
cv_std

In [ ]:
final_cv_score = pd.DataFrame(model_names, columns = ['Regressors'])
final_cv_score['RMSE_mean'] = cv_scores
final_cv_score['RMSE_std'] = cv_std
final_cv_score

In [ ]:
# Train-Test split the data
x_train, x_validation, y_train, y_validation = train_test_split(data, target, test_size = 0.1)

In [ ]:
best_regressor_name = final_cv_score.sort_values(by=['RMSE_mean']).head(1)['Regressors'].iloc[0]
best_regressor = models[model_names.index(best_regressor_name)]
best_regressor

In [ ]:
# The Best Regressor
best_model = best_regressor.fit(x_train, y_train)

In [ ]:
# Creation of the RMSE metric:    
def rmse(y, y_pred):
    return np.sqrt(mean_squared_error(y, y_pred))

In [ ]:
pred = best_model.predict(x_validation)
score = rmse(y_validation, pred)
score

# 4.Submission

In [ ]:
test_pred = best_model.predict(df_test)
submission = pd.DataFrame(test_id, columns = ['seq_id'])
# test_pred = np.expm1(test_pred)
submission['tm'] = test_pred 
submission.head()

In [ ]:
submission.to_csv("submission.csv", index = False, header = True)